In [ ]:
# TODO

# Morgan-Mercer-Floden

In [319]:
#imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from ipywidgets import interact, FloatSlider, Output, VBox
import ipywidgets as widgets
from IPython.display import clear_output, display


In [320]:
#Gompertz model

# Model
def Gompertz(t, k1, k2, k3, k4, k5, k6):
    N_0 = N[0]
    return np.log(N_0) + k1 * np.exp(-np.exp(-k2 * (t - k3))) - k4 * np.exp(-np.exp(-k5 * (t - k6)))
    
# Estimate initial values for Gompertz
def estimate_initial_gompertz(t, N):
    N_0 = N[0]
    N_max = np.max(N)

    k1_init = np.log(N_max / N_0)
    k2_init = 1 / (t[np.argmax(np.diff(N))] - t[0])
    k3_init = t[np.argmax(np.diff(N))] / 2
    k4_init = k1_init / 2
    k5_init = k2_init
    k6_init = k3_init
    return [k1_init, k2_init, k3_init, k4_init, k5_init, k6_init]
    

In [321]:
#Churchill

# Model
def Churchill(t, k1, k2, k3, k4):
    return ((1 / k1) * np.exp(k3 * t) + (1 / k2) * np.exp(k4 * t)) ** (-1)

# Estimate initial values for Churchill
def estimate_initial_churchill(t, N):
    N_0 = N[0]
    N_max = np.max(N)
    if N_max == N_0:
        k1_init = 1.0
        k2_init = 1.0
    else:
        k1_init = 1 / (N_max - N_0)
        k2_init = 1 / (N_max - N_0)
    growth_phase = t[:len(t)//2]
    decay_phase = t[len(t)//2:]
    if np.ptp(growth_phase) == 0 or np.ptp(decay_phase) == 0:
        k3_init = 1.0
        k4_init = 1.0
    else:
        k3_init = np.log(N_max / N_0) / np.ptp(growth_phase)
        k4_init = np.log(N_max / N_0) / np.ptp(decay_phase)
    return [k1_init, k2_init, k3_init, k4_init]


In [322]:
# Weibull 
# https://www.researchgate.net/publication/37628953_Weibull_Distributions_and_Their_Applications

# Model
def Weibull(t, k1, k2, k3):
    return k1 * np.exp(-(t / k2) ** k3)

# Estimate initial values for Weibull
def estimate_initial_weibull(t, N):
    N_max = np.max(N)
    N_min = np.min(N)
    k1_init = N_max
    k2_init = np.mean(t)
    k3_init = 1.0
    return [k1_init, k2_init, k3_init]

In [328]:
# Generalized Logistic Model
# (Richard's curve) Modeled for initial condition adjustment (C instead of 1)
# https://www.researchgate.net/publication/341220486_A_Generalized_Logistic_Function_and_Its_Applications

# Model
def generalized_logistic(t, A, K, B, v, Q, C):
    return A + (K - A) / ((C + Q * np.exp(-B * t)) ** (1/v))

# Estimate initial values for GLM
def estimate_initial_glm(t, N):
    A_init = np.min(N)
    K_init = np.max(N)
    B_init = 1.0
    v_init = 1.0
    Q_init = 1.0
    C_init = 1.0
    return [A_init, K_init, B_init, v_init, Q_init, C_init]

In [323]:
# Richard's curve
# Same base model as above, different names for clarity of use. Modeled for Inflection Point(M)
# https://www.researchgate.net/publication/330083763_A_new_modified_logistic_growth_model_for_empirical_use/fulltext/5c2cbeb5a6fdccfc70780874/A-new-modified-logistic-growth-model-for-empirical-use.pdf

# Model
def Richards(t, A, K, B, v, Q, M):
    return A + (K - A) / ((1 + Q * np.exp(-B * (t - M))) ** (1/v))
# Estimate initial values for Richards
def estimate_initial_richards(t, N):
    A_init = np.min(N)
    K_init = np.max(N)
    
    # Estimate M as the time where the derivative is highest (inflection point)
    diffs = np.diff(N)
    max_diff_idx = np.argmax(diffs)
    M_init = t[max_diff_idx]
    
    # Estimate B based on the steepest slope
    B_init = np.abs(diffs[max_diff_idx] / (N[max_diff_idx+1] - N[max_diff_idx]))
    
    v_init = 1.0
    Q_init = 1.0
    C_init = 1.0
    return [A_init, K_init, B_init, v_init, Q_init, M_init]

In [324]:
# Von Bertalanffy Growth model
# https://www.researchgate.net/figure/Distribution-of-residuals-from-fit-of-von-Bertalanffy-model-Equations-1-3-n3968_fig1_249283955

# Model
def VonBertalanffy(t, L0, L_inf, k):
    return L_inf - (L_inf - L0) * np.exp(-k * t)

# Estimate initial values for Von Bertalanffy
def estimate_initial_vonbertalanffy(t, N):
    L0_init = np.max(N)
    L_inf_init = np.min(N)
    # Estimate k based on the rate of change
    diffs = np.diff(N)
    max_diff_idx = np.argmax(diffs)
    k_init = np.abs(diffs[max_diff_idx] / (t[max_diff_idx+1] - t[max_diff_idx]))
    return [L0_init, L_inf_init, k_init]

In [326]:
# Logistic Growth Model
# https://www.researchgate.net/publication/330083763_A_new_modified_logistic_growth_model_for_empirical_use

# Model
def Logistic(t, k1, k2, k3):
    return k1 / (1 + np.exp(-k2 * (t - k3)))

# Estimate initial values for Logistic
def estimate_initial_logistic(t, N):
    N_max = np.max(N)
    N_min = np.min(N)
    k1_init = N_max - N_min
    k2_init = 1.0
    k3_init = np.median(t)
    return [k1_init, k2_init, k3_init]

In [325]:
# Double exponential Decay
# https://www.researchgate.net/figure/Double-exponential-decay-curves-describing-the-decline-of-residual-organic-N-of-input_fig2_237641766

# Model
def double_exponential(t, a1, b1, a2, b2):
    return a1 * np.exp(-b1 * t) + a2 * np.exp(-b2 * t)

# Estimate initial values for Double Exponential Decay
def estimate_initial_double_exponential(t, N):
    a1_init = N.iloc[0]
    b1_init = 0.1
    a2_init = N.iloc[-1]
    b2_init = 0.01
    return [a1_init, b1_init, a2_init, b2_init]

In [348]:
# Double logistic model

# Model
def double_logistic(t, K1, r1, t1, K2, r2, t2):
    return K1 / (1 + np.exp(-r1 * (t - t1))) + K2 / (1 + np.exp(-r2 * (t - t2)))

# Estimate initial values for Double logistic Model
def estimate_initial_double_logistic(t, N):
    N_max = np.max(N)
    N_min = np.min(N)
    
    # Split data into two halves and find midpoints for initial guesses
    t_split = len(t) // 2
    t1_init = t[t_split // 2]
    t2_init = t[t_split + t_split // 2]
    
    # Estimate K1 and K2 as portions of the maximum and minimum values
    K1_init = (N_max + N_min) / 2
    K2_init = (N_max - N_min) / 2
    
    # Estimate r1 and r2 based on the steepest slopes in the two halves of the data
    diffs1 = np.diff(N[:t_split])
    diffs2 = np.diff(N[t_split:])
    r1_init = np.abs(diffs1[np.argmax(np.abs(diffs1))]) / (t[1] - t[0])
    r2_init = np.abs(diffs2[np.argmax(np.abs(diffs2))]) / (t[1] - t[0])
    
    # Ensure that initial rates are not too large to avoid overflow
    r1_init = min(r1_init, 0.1)
    r2_init = min(r2_init, 0.1)
    
    return [K1_init, r1_init, t1_init, K2_init, r2_init, t2_init]

In [350]:
# Plotting function
def plot_fit(k1, k2, k3, k4, k5, k6, func, num_params):
    plt.scatter(t, N, label='Data')
    if func in [Richards, generalized_logistic, double_logistic]:
        plt.plot(t, func(t, k1, k2, k3, k4, k5, k6), 'r-', label='Fit')
    elif func == Gompertz:
        plt.plot(t, np.exp(func(t, k1, k2, k3, k4, k5, k6)), 'r-', label='Fit')
    elif func in [Churchill,double_exponential]:
        plt.plot(t, func(t, k1, k2, k3, k4), 'r-', label='Fit')
    elif func in [Weibull, Logistic, Logistic_decay, VonBertalanffy, Monod]:
        plt.plot(t, func(t, k1, k2, k3), 'r-', label='Fit')
    plt.xlim(0, 160)
    plt.ylim(0, 8)
    plt.xticks(np.arange(0, 180, step=20))
    plt.xlabel('t (days)')
    plt.ylabel('log(CFU/ml)')
    plt.legend()
    plt.show()

# Processes file and launches interactables
def process_file(file_name, func):
    clear_output(wait=True)
    global t, N
    data = pd.read_csv(file_name, header=None)
    t = data[0]
    N = data[1]

    # Initializing parameters
    if func == Gompertz:
        p0 = estimate_initial_gompertz(t, N)
        num_params = 6
    elif func == Churchill:
        p0 = estimate_initial_churchill(t, N)
        num_params = 4
    elif func == Logistic:
        p0 = estimate_initial_logistic(t, N)
        num_params = 3
    elif func == generalized_logistic:
        p0 = estimate_initial_glm(t, N)
        num_params = 6
    elif func == Weibull:
        p0 = estimate_initial_weibull(t, N)
        num_params = 3
    elif func == Richards:
        p0 = estimate_initial_richards(t, N)
        num_params = 6
    elif func == VonBertalanffy:
        p0 = estimate_initial_vonbertalanffy(t, N)
        num_params = 3
    elif func == double_exponential:
        p0 = estimate_initial_double_exponential(t, N)
        num_params = 4
    elif func == double_logistic:
        p0 = estimate_initial_double_logistic(t,N)
        num_params = 6
    else:
        p0 = [1, 1, 1, 1, 1, 1]
        num_params = 6

    print("Initial parameter estimates:", p0)

    try:
        if func in [Weibull, Logistic, Logistic_decay, Richards, VonBertalanffy, double_exponential,generalized_logistic]:
            popt, pcov = curve_fit(func, t, N, p0=p0[:num_params], bounds=(0, np.inf), maxfev=9999)
        elif func in [double_logistic]:
            popt, pcov = curve_fit(func, t, N, p0=p0[:num_params], bounds=(0, [100, 1, np.max(t), 100, 1, np.max(t)]), maxfev=9999)
        else:
            popt, pcov = curve_fit(func, t, np.log(N), p0=p0[:num_params], bounds=(0, np.inf), maxfev=9999)
        print("Fitted parameters:", popt)
    except RuntimeError as e:
        print(f"An error occurred during fitting: {e}")
        return
    except ValueError as e:
        print(f"Value error during fitting: {e}")
        return

    sliders = [FloatSlider(min=0.01, max=100, step=0.001, value=param, description=f'k{i+1}', continuous_update=False) for i, param in enumerate(popt[:num_params])]

    out = Output()

    def update_plot(**kwargs):
        with out:
            clear_output(wait=True)
            plot_fit(kwargs.get('k1', 0), kwargs.get('k2', 0), kwargs.get('k3', 0), kwargs.get('k4', 0), kwargs.get('k5', 0), kwargs.get('k6', 0), func, num_params)

    interact(update_plot, **{f'k{i+1}': sliders[i] for i in range(num_params)})

    box = VBox([out], layout=widgets.Layout(height='500px'))
    display(box)

# Example usage
# process_file('your_data_file.csv', Gompertz)

In [351]:
process_file("control.csv", Gompertz)

Initial parameter estimates: [1.2568170281019202, 0.3333333333333333, 1.5, 0.6284085140509601, 0.3333333333333333, 1.5]
Fitted parameters: [ 1.17643862  2.24115774  6.25559615  0.2449459   0.07034968 68.26995583]


interactive(children=(FloatSlider(value=1.1764386217062868, continuous_update=False, description='k1', min=0.0…

In [353]:
process_file('B281.csv', double_logistic)

Initial parameter estimates: [6.59, 0.5166666666666666, 31, 1.3849999999999998, 0.13833333333333334, 111]
Fitted parameters: [6.29586840e+00 9.99999694e-01 2.13201189e-10 3.74823688e-10
 5.35166118e-01 1.44909684e+02]


interactive(children=(FloatSlider(value=6.2958683984302475, continuous_update=False, description='k1', min=0.0…